## Tranformer
### Let's build the core encoder and decoder model for machine translation

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math

In [2]:
## Embedding
class inputEmbedding(nn.Module):
  def __init__(self,vocab_size,embedding_dim):
    super().__init__()
    self.vocab_size = vocab_size
    self.embedding_dim = embedding_dim
    self.embedding = nn.Embedding(vocab_size,embedding_dim)
  def forward(self,x):
    x = self.embedding(x)
    return x

In [3]:
# Test
batch_size = 32
sequence_length = 100
vocab_size = 5000
embedding_dim = 128
x = torch.ones(batch_size,sequence_length,dtype=torch.long)
input = inputEmbedding(vocab_size,embedding_dim)
input(x).shape

torch.Size([32, 100, 128])

In [4]:
## Positional Encoding
class positionalEncoding(nn.Module):
  def __init__(self,seq_len,d_k):
    super().__init__()
    self.seq_len = seq_len
    self.d_k = d_k
    pe = torch.zeros(seq_len,d_k)
    div = torch.exp(torch.arange(0,self.d_k,2).float() * - (math.log(10000.0)/self.d_k))
    positions = torch.arange(0,self.seq_len,dtype=torch.float).unsqueeze(1)
    pe[:,0::2] = torch.sin(positions*div)
    pe[:,1::2] = torch.cos(positions*div)
    pe = pe.unsqueeze(0)
    self.register_buffer("pe",pe)
  def forward(self,x):
    return x + self.pe[:,:self.seq_len,:].requires_grad_(False)

In [5]:
## test
x = torch.zeros(32,100,128,dtype=torch.float)
pos = positionalEncoding(sequence_length,embedding_dim)
pos(x).shape

torch.Size([32, 100, 128])

In [6]:
## Feed Forward Network
class FeedForward(nn.Module):
  def __init__(self,d_k):
    super().__init__()
    self.d_k = d_k
    self.fnn = nn.Sequential(
        nn.Linear(self.d_k,2048),
        nn.ReLU(),
        nn.Linear(2048,self.d_k)
    )
  def forward(self,x):
    return self.fnn(x)

In [7]:
## test
x = torch.zeros(32,100,128)
fnn_ = FeedForward(embedding_dim)
fnn_(x).shape

torch.Size([32, 100, 128])

In [8]:
## layer norm
class layerNorm(nn.Module):
  def __init__(self,embedding_dim):
    super().__init__()
    self.alpha = nn.Parameter(torch.ones(embedding_dim))
    self.beta = nn.Parameter(torch.ones(embedding_dim))
    self.eps = 0.001
    pass
  def forward(self,x):
    mean = x.mean(dim=-1, keepdim=True)
    std = x.std(dim=-1, keepdim=True)
    x = self.alpha * (x-mean)/(self.eps+std) + self.beta
    return x

In [9]:
ln = layerNorm(128)
ln(x).shape

torch.Size([32, 100, 128])

In [32]:
## Multihead Self Attention
class MultiHeadAttention(nn.Module):
  def __init__(self,d_k,heads,head_dim,device):
    super().__init__()
    self.d_k = d_k
    assert d_k == heads*head_dim
    self.heads = heads
    self.head_dim = head_dim
    self.Q = nn.Linear(self.d_k,self.d_k)
    self.K = nn.Linear(self.d_k,self.d_k)
    self.V = nn.Linear(self.d_k,self.d_k)
    self.device = device
    self.out = nn.Linear(self.d_k,self.d_k)
  def forward(self,x,mask):
    B,S,E = x.size()
    q = self.Q(x).view(B,S,self.heads,self.head_dim).transpose(1,2) # [32,100,128] => [32,100,4,32]
    k = self.K(x).view(B,S,self.heads,self.head_dim).transpose(1,2)
    v = self.V(x).view(B,S,self.heads,self.head_dim).transpose(1,2)
    k = k.transpose(-2,-1) # [32,4,100,32]
    attention_scores = ((q @ k)/math.sqrt(self.d_k))
    if mask :
      mat = torch.tril(torch.ones(S,S)).to(self.device)
      mat = mat.unsqueeze(0).unsqueeze(0)
      attention_scores = attention_scores.masked_fill(mat == 0,-1e9) ## where where their is 0 put to -infinity
    attention_scores = torch.softmax(attention_scores,dim=-1) @ v
    attention_score = attention_scores.transpose(1,2).contiguous().view(B,S,E)
    return self.out(attention_score)

In [33]:
att = MultiHeadAttention(128,4,32,"cpu")
x = torch.zeros(32,100,128)
att(x,1).shape

torch.Size([32, 100, 128])

In [36]:
## cross attention
class CrossAttention(nn.Module):
  def __init__(self,d_k,heads,head_dim):
    super().__init__()
    self.d_k = d_k
    assert d_k == heads*head_dim
    self.heads = heads
    self.head_dim = head_dim
    self.Q = nn.Linear(self.d_k,self.d_k)
    self.out = nn.Linear(self.d_k,self.d_k)
  def forward(self,x,k,v):
    B,S,E = x.size()
    q = self.Q(x).view(B,S,self.heads,self.head_dim).transpose(1,2)
    k = k.view(B,S,self.heads,self.head_dim).transpose(1,2)
    v = v.view(B,S,self.heads,self.head_dim).transpose(1,2)
    k = k.transpose(-2,-1) # [32,4,100,32]
    attention_scores = ((q @ k)/math.sqrt(self.d_k))
    attention_scores = torch.softmax(attention_scores,dim=-1) @ v
    attention_score = attention_scores.transpose(1,2).contiguous().view(B,S,E)
    return self.out(attention_score)

In [37]:
cr = CrossAttention(128,4,32)
cr(x,x,x).shape

torch.Size([32, 100, 128])

In [39]:
## residual layer
class residualLayer(nn.Module):
  def __init__(self):
    super().__init__()
    self.norm = layerNorm()
    self.dropout = nn.Dropout(0.1)
  def forward(self,x,sub_layer):
    return self.norm(x + self.dropout(sub_layer))

## Lets build the encoder and decoder class

In [49]:
class EncoderBlock(nn.Module):
  def __init__(self,attention:MultiHeadAttention,fnn:FeedForward,d_k):
    super().__init__()
    self.attention = attention
    self.attention = attention
    self.fnn = fnn
    self.norm = layerNorm(d_k)
  def forward(self,x):
    # here the x coems is positionally encoded
    x = self.norm(x + self.attention(x,0))
    x = self.norm(x + self.fnn(x))
    return x

In [50]:
enc = EncoderBlock(att,fnn_,128)
enc(x).shape

torch.Size([32, 100, 128])

In [72]:
class DecoderBlock(nn.Module):
  def __init__(self,attention:MultiHeadAttention,fnn:FeedForward,cr:CrossAttention,d_k):
    super().__init__()
    self.attention = attention
    self.attention = attention
    self.fnn = fnn
    self.norm = layerNorm(d_k)
    self.cr = cr
    # self.k = nn.Linear(d_k,d_k)
    # self.V = nn.Linear(d_k,d_k)
  def forward(self,x,k,v):
    # here the x coems is positionally encoded
    x = self.norm(x + self.attention(x,1))
    x = self.norm(x + self.cr(x,k,v))
    x = self.norm(x + self.fnn(x))
    return x

In [73]:
dec = DecoderBlock(att,fnn_,cr,128)
dec(x,x,x).shape

torch.Size([32, 100, 128])

In [56]:
class InitialLayer(nn.Module):
  def __init__(self,embedding:inputEmbedding,positional:positionalEncoding):
    super().__init__()
    self.embedding = embedding
    self.pos = positional
  def forward(self,x):
    x = self.pos(self.embedding(x))
    return x

In [60]:
it = InitialLayer(input,pos)
y = torch.zeros(32,100).long()
it(y).shape

torch.Size([32, 100, 128])

In [65]:
class OutputLayer(nn.Module):
  def __init__(self,d_k,vocab_size):
    super().__init__()
    self.d_k = d_k
    self.out = nn.Linear(d_k,vocab_size)
  def forward(self,x):
    return self.out(x) # cross entropy as softmax

In [66]:
out = OutputLayer(128,1000)
out(x).shape

torch.Size([32, 100, 1000])

## Encoder and decoder

In [68]:
from torch.nn.modules import ModuleList
class Encoder(nn.Module):
  def __init__(self,sublist:nn.ModuleList,d_k):
    super().__init__()
    self.sublayer = sublist
    self.norm = layerNorm(d_k)
  def forward(self,x):
    for layer in self.sublayer:
      x = layer(x)
    return self.norm(x)

In [74]:
from torch.nn.modules import ModuleList
class Decoder(nn.Module):
  def __init__(self,sublist:nn.ModuleList,d_k):
    super().__init__()
    self.sublayer = sublist
    self.norm = layerNorm(d_k)
  def forward(self,x,k,v):
    for layer in self.sublayer:
      x = layer(x,k,v)
    return self.norm(x)

## Transformer

In [75]:
class Transformer(nn.Module):
  def __init__(self,encoder:Encoder,decoder:Decoder,initial:InitialLayer,out:OutputLayer,d_k):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.out = out
    self.initial = initial
    self.K = nn.Linear(d_k,d_k)
    self.V = nn.Linear(d_k,d_k)
  def forward(self,x,y):
    x = self.initial(x)
    y = self.initial(y)
    x = self.encoder(x)
    k,v = self.K(x),self.V(x)
    y = self.decoder(y,k,v)
    y = self.out(y)
    return y

## initialization

In [76]:
# --- Parameters ---
d_k = 128
heads = 4
head_dim = 32
v_size = 5000
seq_len = 100
device = "cpu"

# 1. Basic Components
emb = inputEmbedding(v_size, d_k)
pos = positionalEncoding(seq_len, d_k)
it = InitialLayer(emb, pos)
fnn = FeedForward(d_k)
att = MultiHeadAttention(d_k, heads, head_dim, device)
cr = CrossAttention(d_k, heads, head_dim)
out_layer = OutputLayer(d_k, v_size)

# 2. Build Stacks using ModuleList
# We create 6 unique layers for Encoder and Decoder
enc_blocks = nn.ModuleList([EncoderBlock(att, fnn, d_k) for _ in range(6)])
dec_blocks = nn.ModuleList([DecoderBlock(att, fnn, cr, d_k) for _ in range(6)])

encoder_stack = Encoder(enc_blocks, d_k)
decoder_stack = Decoder(dec_blocks, d_k)

# 3. Initialize Transformer
model = Transformer(encoder_stack, decoder_stack, it, out_layer, d_k)

# 4. Testing with Zeros Tensor
# Input: (Batch, Seq_Len)
src = torch.zeros((32, 100), dtype=torch.long)
trg = torch.zeros((32, 100), dtype=torch.long)

output = model(src, trg)

print(f"Final Output Shape: {output.shape}")
# Expected: torch.Size([32, 100, 5000])

Final Output Shape: torch.Size([32, 100, 5000])
